# musicar: MIDI from the released models

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/DmytroHilei/Music_generative_model/blob/transformer-v2/notebooks/musicar_colab.ipynb)

The musicar models are autoregressive transformers that write symbolic music (MIDI) one note at a time: each step
picks a note's time shift, instrument, pitch, duration and velocity. This notebook installs the
[code](https://github.com/DmytroHilei/Music_generative_model), downloads a model from the Hugging Face Hub and
generates from scratch, continues a MIDI file of yours, or follows a text caption (musicar-L-caption). Each result is
rendered to audio with fluidsynth, drawn as a piano roll and offered for download.

| Model | Parameters | Plays | Context | Use it for |
|---|---|---|---|---|
| [musicar-L](https://huggingface.co/Dimitri-206265/musicar-L) | 464M | 128 General MIDI instruments + drums | 2,048 notes | the main pretrained model: the best continuations |
| [musicar-S](https://huggingface.co/Dimitri-206265/musicar-S) | 108M | 128 General MIDI instruments + drums | 2,048 notes | a small, fast multi-instrument model |
| [musicar-S-piano](https://huggingface.co/Dimitri-206265/musicar-S-piano) | 107M | solo piano | 512 notes | solo piano |
| [musicar-L-caption](https://huggingface.co/Dimitri-206265/musicar-L-caption) | 477M + the LaBSE text encoder (471M) | 128 General MIDI instruments + drums | 2,048 notes | text to MIDI: a caption in English or Ukrainian and a length in seconds |

**Runtime.** A free T4 GPU is recommended for musicar-L and musicar-L-caption (Runtime → Change runtime type → T4
GPU). The S models also run on the CPU runtime: about 25-45 notes per second on two CPU threads of a laptop.

**License.** The models are released under [CC BY-NC-SA 4.0](https://creativecommons.org/licenses/by-nc-sa/4.0/)
(non-commercial, share-alike), because their training data are under non-commercial terms. The model cards describe
the data, the training and the evaluation.

## 1. Install

The `musicar` package from GitHub, with the `[text]` extra (the `transformers` package, for musicar-L-caption's
LaBSE encoder), and matplotlib for the piano roll. For audio, fluidsynth and the FluidR3 General MIDI soundfont; on
Colab the cell installs them, elsewhere install them yourself (Debian / Ubuntu: `sudo apt install fluidsynth
fluid-soundfont-gm`) or go without audio.

In [ ]:
%pip install -q "musicar[text] @ git+https://github.com/DmytroHilei/Music_generative_model" matplotlib

import shutil, subprocess, sys
if "google.colab" in sys.modules and not shutil.which("fluidsynth"):
    subprocess.run("apt-get -qq update > /dev/null && apt-get -qq install -y fluidsynth fluid-soundfont-gm > /dev/null",
                   shell=True, check=False)
print("fluidsynth:", shutil.which("fluidsynth") or "not installed (no audio: MIDI files and piano rolls only)")

## 2. Listening tools

`listen(midi, name)` saves a result to `musicar_out/<name>.mid`, renders it to a WAV file with fluidsynth and plays
it, draws its piano roll and gives a download link for the MIDI file. The generation cells below call it: run this
cell once.

In [ ]:
#@title Listening tools (run once) { display-mode: "form" }
import base64, html, os, shutil, subprocess, tempfile, time, wave

import matplotlib.pyplot as plt
import numpy as np
import pretty_midi
from IPython.display import Audio, HTML, display

SOUNDFONT = "/usr/share/sounds/sf2/FluidR3_GM.sf2"  # from fluid-soundfont-gm
OUT = "musicar_out"  # this session's MIDI and WAV files
RATE = 32000  # audio sample rate
results = {}  # name -> PrettyMIDI, filled by listen()


def render_wav(midi_path, wav_path):
    """The MIDI as a stereo WAV file through fluidsynth, peaks at -1 dBFS; False without fluidsynth or the soundfont.
    The render stops at the end of the music + 3 s: fluidsynth's fast render has been seen to run far past the end
    of a file."""
    if not shutil.which("fluidsynth") or not os.path.exists(SOUNDFONT):
        return False
    floats = int((pretty_midi.PrettyMIDI(midi_path).get_end_time() + 3) * RATE) * 2
    with tempfile.TemporaryDirectory() as tmp:
        raw = os.path.join(tmp, "audio.raw")
        proc = subprocess.Popen(["fluidsynth", "-niq", "-r", str(RATE), "-T", "raw", "-O", "float", "-F", raw,
                                 SOUNDFONT, midi_path], stdout=subprocess.DEVNULL, stderr=subprocess.DEVNULL)
        deadline = time.time() + 120
        while proc.poll() is None:
            if (os.path.exists(raw) and os.path.getsize(raw) >= 4 * floats) or time.time() > deadline:
                proc.kill()
            time.sleep(0.05)
        audio = np.fromfile(raw, dtype="<f4")[:floats] if os.path.exists(raw) else np.zeros(0, "<f4")
    peak = float(np.abs(audio).max(initial=0.0))
    if peak < 1e-4:  # no notes: only fluidsynth's noise floor, which normalising would make a loud thump
        return False
    pcm = (audio[:audio.size // 2 * 2] * (0.89 / peak) * 32767).astype("<i2")
    with wave.open(wav_path, "wb") as w:
        w.setnchannels(2)
        w.setsampwidth(2)
        w.setframerate(RATE)
        w.writeframes(pcm.tobytes())
    return True


# piano roll colours: the three busiest instruments, every other instrument in grey, drums in a lane below
SURFACE, INK, MUTED, GRID, AXIS = "#fcfcfb", "#52514e", "#898781", "#e1e0d9", "#c3c2b7"
SERIES, OTHER = ("#2a78d6", "#eb6834", "#1baf7a"), "#b4b2aa"


def piano_roll(midi):
    pitched = sorted((i for i in midi.instruments if i.notes and not i.is_drum), key=lambda i: -len(i.notes))
    hits = [n.start for i in midi.instruments if i.is_drum for n in i.notes]
    if hits:
        fig, (ax, lane) = plt.subplots(2, 1, figsize=(10, 4.4), sharex=True, facecolor=SURFACE, layout="constrained",
                                       gridspec_kw=dict(height_ratios=[6, 1]))
    else:
        (fig, ax), lane = plt.subplots(figsize=(10, 3.8), facecolor=SURFACE, layout="constrained"), None
    groups = [(SERIES[k], pretty_midi.program_to_instrument_name(i.program), i.notes) for k, i in enumerate(pitched[:3])]
    rest = [n for i in pitched[3:] for n in i.notes]
    if rest:
        groups.append((OTHER, f"{len(pitched) - 3} more instruments", rest))
    for color, label, notes in reversed(groups):  # the busiest instrument is drawn last, on top
        ax.barh([n.pitch for n in notes], [n.end - n.start for n in notes], left=[n.start for n in notes],
                height=0.8, color=color, edgecolor=SURFACE, linewidth=0.3, label=f"{label} ({len(notes):,})")
    for a in (ax, lane) if lane else (ax,):
        a.set_facecolor(SURFACE)
        for side in ("top", "right", "left"):
            a.spines[side].set_visible(False)
        a.spines["bottom"].set_color(AXIS)
        a.tick_params(colors=MUTED, labelsize=8, length=0)
    if groups:
        lo = min(n.pitch for _, _, notes in groups for n in notes) - 2
        hi = max(n.pitch for _, _, notes in groups for n in notes) + 2
        ticks = [p for p in range(0, 128, 12) if lo <= p <= hi] or [lo + 2, hi - 2]
        ax.set_ylim(lo, hi)
        ax.set_yticks(ticks, [pretty_midi.note_number_to_name(p) for p in ticks])
        ax.yaxis.grid(True, color=GRID, linewidth=0.6)
        ax.set_axisbelow(True)
        handles, labels = ax.get_legend_handles_labels()
        if len(labels) == 1:  # one instrument: a label, no legend
            ax.set_title(labels[0], loc="left", color=INK, fontsize=8)
        else:
            ax.legend(handles[::-1], labels[::-1], loc="lower left", bbox_to_anchor=(0, 1), ncol=4, frameon=False,
                      fontsize=8, labelcolor=INK, handlelength=1, columnspacing=1.2, borderaxespad=0.2)
    else:
        ax.set_yticks([])
        ax.text(0.5, 0.5, "no pitched notes", transform=ax.transAxes, ha="center", color=MUTED)
    if lane:
        lane.vlines(hits, 0.15, 0.85, color=INK, linewidth=0.6)
        lane.set_ylim(0, 1)
        lane.set_yticks([0.5], [f"drums ({len(hits):,})"])
    (lane or ax).set_xlabel("seconds", color=MUTED, fontsize=8)
    ax.set_xlim(0, max(midi.get_end_time(), 1.0))
    return fig


def download_link(path):
    """A link that saves the file from the page (it is embedded in the link: fine for MIDI files)."""
    name = html.escape(os.path.basename(path))
    data = base64.b64encode(open(path, "rb").read()).decode()
    return HTML(f'<a download="{name}" href="data:audio/midi;base64,{data}" style="display:inline-block;'
                'margin-top:6px;padding:6px 14px;border-radius:6px;background:#2a78d6;color:#fff;'
                f'font:14px system-ui,sans-serif;text-decoration:none">Download {name}</a>')


def instrument_names(midi):
    return ", ".join(sorted({"Drums" if i.is_drum else pretty_midi.program_to_instrument_name(i.program)
                             for i in midi.instruments if i.notes}))


def listen(midi, name):
    """Save a result as musicar_out/<name>.mid (+ .wav), play it, draw its piano roll, offer the MIDI file."""
    results[name] = midi
    os.makedirs(OUT, exist_ok=True)
    mid = os.path.join(OUT, f"{name}.mid")
    midi.write(mid)
    notes = sum(len(i.notes) for i in midi.instruments)
    print(f"{name}: {notes:,} notes, {midi.get_end_time():.1f} s; instruments: {instrument_names(midi) or 'none'}")
    wav = mid[:-len(".mid")] + ".wav"
    if render_wav(mid, wav):
        display(Audio(wav))
    else:
        print("No audio: fluidsynth or the soundfont is missing (section 1).")
    piano_roll(midi)
    plt.show()
    display(download_link(mid))

## 3. Pick a model

Set the fields and run the cell; run it again after changing any of them (the model is reloaded only when
`model_name` changes). The weights are downloaded from the Hub on first use (0.2 GB for the S models, 0.9 GB for
musicar-L). The captioned model has its own section (6).

- `n_notes`: notes to generate (one note per step); musicar-L can end a piece earlier with its end token.
- `temperature`: lower = safer and more repetitive, higher = more varied. 0.85-0.9 is a good start.
- `seed`: 0 = a new random seed for every run, printed in the run's log (`Generating ... (seed N)`); give that
  number to get the same piece again with the same settings.
- `instruments_mode` (multi-instrument models): `sample` = the model chooses each note's instrument, `piano` = all
  piano, `prompt` = only the instruments of the MIDI file you continue (section 5), `list` = only the General MIDI
  programs in `instrument_list` (0 piano, 25 acoustic guitar, 33 electric bass, 48 strings, 128 = drums).
  musicar-S-piano always plays piano.

In [ ]:
#@title Model and sampling settings
model_name = "musicar-S"  #@param ["musicar-S", "musicar-S-piano", "musicar-L"]
n_notes = 512  #@param {type:"slider", min:64, max:2048, step:64}
temperature = 0.85  #@param {type:"slider", min:0.5, max:1.3, step:0.05}
seed = 0  #@param {type:"integer"}
instruments_mode = "sample"  #@param ["sample", "piano", "prompt", "list"]
instrument_list = "0, 33, 25, 128"  #@param {type:"string"}

import torch
import musicar

DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
# bf16 needs an Ampere or newer GPU: on the free T4 (compute capability 7.5) and on the CPU the models run in float32
DTYPE = "bf16" if DEVICE == "cuda" and torch.cuda.get_device_capability()[0] >= 8 else "fp32"

if globals().get("model") is None or globals().get("loaded") != model_name:
    model, loaded = None, None  # free the previous model before loading the next
    if DEVICE == "cuda":
        torch.cuda.empty_cache()
    model = musicar.from_pretrained(model_name, device=DEVICE, dtype=DTYPE)
    loaded = model_name
print(f"{model_name} on {DEVICE} ({DTYPE})")


def instruments(has_prompt=False):
    """generate()'s instruments= for instruments_mode."""
    if not model.config.n_programs:
        return "piano"  # musicar-S-piano
    if instruments_mode == "list":
        return [int(p) for p in instrument_list.replace(",", " ").split()]
    if instruments_mode == "prompt" and not has_prompt:
        return "sample"  # no MIDI file to take them from
    return instruments_mode


def seed_arg(value):
    """generate()'s seed=: None (a random seed, printed in the log) for 0 or less."""
    return value if value > 0 else None

## 4. Generate from scratch

musicar-L starts from its start-of-piece token. The S models have none and start from one artificial seed note, so
their openings can wander, and musicar-S can get very dense (hundreds of notes in a few seconds): continuing a MIDI
file (section 5) or fewer notes sound better. musicar-S picks several instruments on its own; from scratch musicar-L
mostly stays with the instrument of its first note, usually piano (an instrument list mostly decides which
one), so for a band, continue a band's MIDI file.

In [ ]:
midi = musicar.generate(model, n_notes=n_notes, instruments=instruments(), temperature=temperature,
                        seed=seed_arg(seed))
listen(midi, "from_scratch")

## 5. Continue a MIDI file

The model hears the first `prompt_notes` notes of the file and continues them; with `keep_prompt` the result starts
with those notes. On Colab, leave `midi_path` empty and choose a file when the cell asks; elsewhere set `midi_path`.
The sampling settings are section 3's.

In [ ]:
#@title Continue a MIDI file
midi_path = ""  #@param {type:"string"}
prompt_notes = 64  #@param {type:"slider", min:16, max:512, step:16}
keep_prompt = True  #@param {type:"boolean"}

import sys
if not midi_path and "google.colab" in sys.modules:
    from google.colab import files
    uploaded = files.upload()  # saved next to the notebook
    midi_path = next(iter(uploaded), "")
if not midi_path:
    print("No MIDI file: on Colab run the cell again and choose one; elsewhere set midi_path to a .mid file.")
else:
    midi = musicar.generate(model, n_notes=n_notes, prompt=midi_path, prompt_notes=prompt_notes,
                            instruments=instruments(has_prompt=True), temperature=temperature, seed=seed_arg(seed),
                            keep_prompt=keep_prompt)
    listen(midi, "continuation")

## 6. Text to MIDI: musicar-L-caption

The captioned model reads a caption in English or Ukrainian and a length in seconds. It follows broad attributes
(pace, loudness, register, piano / no drums / strings / guitar, sad or happy), not reliably genre, key or exact
instruments. It runs with the `f1` preset, the decoding settings tuned for it (guidance 2.5, temperature 1.1, a
repetition penalty, the end placed near `length_seconds`), so section 3's temperature and instruments are not used.
It needs the `[text]` extra (section 1); the first run downloads the model (about 1 GB) and LaBSE (about 1.9 GB).
A GPU is recommended: on the CPU a 60-second piece takes several minutes. `caption_seed` works like section 3's
`seed`.

In [ ]:
#@title Generate from a caption
caption = "a calm jazz piano trio"  #@param {type:"string"}
length_seconds = 60  #@param {type:"slider", min:10, max:120, step:5}
caption_seed = 0  #@param {type:"integer"}
run_on_cpu = False  #@param {type:"boolean"}
# Ukrainian works as well. "solo" with one instrument, "only ..." or "without ..." are strict limits on the
# instruments, e.g. "сумна повільна балада, фортепіано соло" (a sad slow ballad, solo piano)

if DEVICE == "cpu" and not run_on_cpu:
    print("Skipped: this runtime has no GPU. Switch to a T4 GPU (Runtime → Change runtime type and run sections 1-3 "
          "again), or tick run_on_cpu.")
else:
    if globals().get("caption_model") is None:
        caption_model = musicar.from_pretrained("musicar-L-caption", device=DEVICE, dtype=DTYPE)
    # n_notes is only an upper bound here: the piece ends near length_seconds with the model's end token
    midi = musicar.generate(caption_model, text=caption, preset="f1", length_seconds=length_seconds, n_notes=4096,
                            seed=seed_arg(caption_seed))
    listen(midi, "caption")

## 7. Listen again and download

Plays any result of this session again with its piano roll and MIDI link. On Colab, `download_files` also saves its
MIDI and WAV files to your computer; they are in the `musicar_out` folder of the Files panel as well.

In [ ]:
#@title Listen again / download
which = "from_scratch"  #@param ["from_scratch", "continuation", "caption"]
download_files = False  #@param {type:"boolean"}

import sys
if which not in results:
    print(f"No {which} result yet (results so far: {', '.join(results) or 'none'}).")
else:
    listen(results[which], which)
    if download_files and "google.colab" in sys.modules:
        from google.colab import files
        for ext in (".mid", ".wav"):
            if os.path.exists(os.path.join(OUT, which + ext)):
                files.download(os.path.join(OUT, which + ext))